# Week 10 — Ridge Regression from Scratch (NumPy)

**Goal:** Ridge (L2) closed-form and GD; compare OLS vs ridge on collinear features; shrinkage vs lambda; train/test MSE curves.

No scikit-learn for the core.


## 1. Collinear data

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from ridge_regression_scratch import (
    make_collinear_regression, train_test_split_xy, RidgeRegression, ols_fit,
    mse, lambda_sweep, plot_results
)

X, y, true_coef = make_collinear_regression(n_samples=150, noise=0.6, seed=42)
X_train, X_test, y_train, y_test = train_test_split_xy(X, y, test_size=0.3, seed=1)
print("true:", true_coef)


## 2. OLS vs Ridge

In [ ]:
ols = ols_fit(X_train, y_train)
ridge = RidgeRegression(alpha=10.0, method="closed_form").fit(X_train, y_train)
print("OLS  ", ols.coef_.round(4), "test MSE", round(mse(y_test, ols.predict(X_test)), 4))
print("Ridge", ridge.coef_.round(4), "test MSE", round(mse(y_test, ridge.predict(X_test)), 4))


## 3. Lambda sweep

In [ ]:
alphas = np.logspace(-2, 3, 40)
sweep = lambda_sweep(X_train, y_train, X_test, y_test, alphas)
best = int(np.argmin(sweep["test_mse"]))
print("best lambda", alphas[best], "test MSE", sweep["test_mse"][best])
plot_results(sweep, ols.coef_, ridge.coef_, true_coef, ROOT / "outputs")
img = plt.imread(ROOT / "outputs" / "ridge_results.png")
plt.figure(figsize=(13, 4)); plt.imshow(img); plt.axis("off"); plt.show()


## Takeaways

- Collinearity inflates OLS coefficient variance; ridge shrinks weights toward zero.
- Larger λ → more bias, less variance; pick λ by validation / test MSE.
- Closed-form and GD agree when GD is run long enough with a sensible step size.
